In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# 1. Đọc dữ liệu
print("Đang đọc dữ liệu...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Tách features và target
X = train.drop(['Academic_Status', 'Student_ID'], axis=1)
y = train['Academic_Status']
X_test = test.drop(['Student_ID'], axis=1)

# --- PHẦN CẢI TIẾN QUAN TRỌNG ---

# 2. Phân loại cột kỹ hơn
# Xác định cột Text (Cần xử lý ngôn ngữ)
text_cols = ['Advisor_Notes', 'Personal_Essay'] 

# Xác định các cột số và cột phân loại còn lại
# (Loại bỏ các cột text ra khỏi danh sách categorical mặc định để tránh lỗi)
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = [col for col in X.select_dtypes(include=['object']).columns 
                    if col not in text_cols]

# Xử lý giá trị thiếu cho cột Text trước khi đưa vào Pipeline
# (Thay NaN bằng chuỗi rỗng để không bị lỗi)
X[text_cols] = X[text_cols].fillna('')
X_test[text_cols] = X_test[text_cols].fillna('')

# 3. Xây dựng bộ xử lý (Transformer)
# - Số: Điền giá trị trung bình
numeric_transformer = SimpleImputer(strategy='median')

# - Phân loại: OneHotEncoder (như cũ)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

text_transformer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))

# Gộp tất cả lại
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
        # Áp dụng TF-IDF cho từng cột text riêng biệt
        ('text1', text_transformer, 'Advisor_Notes'), 
        # ('text2', text_transformer, 'Personal_Essay') # Tạm thời tắt cột này nếu chạy lâu, hoặc mở ra nếu muốn
    ])

# 4. Mô hình XGBoost (Vũ khí hạng nặng)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=500,       # Tăng số lượng cây học
        learning_rate=0.05,     # Học chậm lại để chắc chắn hơn
        max_depth=6,            # Độ sâu của cây
        random_state=42,
        n_jobs=-1               # Dùng hết sức mạnh CPU
    ))
])

# 5. Train và Predict
print("Đang training XGBoost (Sẽ lâu hơn chút xíu vì mô hình xịn hơn)...")
model.fit(X, y)

print("Đang dự đoán...")
predictions = model.predict(X_test)

# 6. Xuất file
submission = pd.DataFrame({
    'Student_ID': test['Student_ID'],
    'Academic_Status': predictions
})
submission.to_csv('submission_v2_xgboost.csv', index=False)
print("Đã tạo file 'submission_v2_xgboost.csv'. ")

# 7. Serialize mô hình dạng pickle
print("Đang lưu mô hình...")
with open('xgboost_pipeline.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Đã lưu thành công file 'xgboost_pipeline.pkl'!")